In [1]:
# =============================================================================
# SCD Sensitivity to K – Geração de TIFF de alta resolução (1200 dpi)
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('agg')  # evita problemas de GUI no Kaggle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')

# Configurações
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

N_SAMPLES = 5000
INPUT_DIM = 50
LATENT_DIM = 16
K_VALUES = [2, 4, 6, 8]
BATCH_SIZE = 128
EPOCHS = 20

# 1. Gerar dados sintéticos
print("Gerando dados...")
X_iid = np.random.rand(N_SAMPLES, INPUT_DIM).astype(np.float32)

K_markov = 4
p_self = 0.95
states = np.zeros(N_SAMPLES, dtype=int)
current = np.random.randint(K_markov)
states[0] = current
for i in range(1, N_SAMPLES):
    if np.random.rand() < p_self:
        current = current
    else:
        new = current
        while new == current:
            new = np.random.randint(K_markov)
        current = new
    states[i] = current
means = np.linspace(-2, 2, K_markov).reshape(-1, 1)
X_markov = np.zeros((N_SAMPLES, INPUT_DIM))
for i, s in enumerate(states):
    X_markov[i] = means[s] + np.random.randn(INPUT_DIM) * 0.5
X_markov = X_markov.astype(np.float32)

# 2. Autoencoder
class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim),
            nn.Sigmoid()
        )
    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon, z

def train_ae(X, input_dim, latent_dim, epochs=20, batch_size=128):
    dataset = TensorDataset(torch.tensor(X))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    model = Autoencoder(input_dim, latent_dim)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()
    model.train()
    for epoch in range(epochs):
        loss_total = 0
        for (x,) in loader:
            optimizer.zero_grad()
            recon, _ = model(x)
            loss = criterion(recon, x)
            loss.backward()
            optimizer.step()
            loss_total += loss.item()
        if epoch % 5 == 0:
            print(f"Epoch {epoch}, loss: {loss_total/len(loader):.4f}")
    return model

print("Treinando AE para i.i.d...")
model_iid = train_ae(X_iid, INPUT_DIM, LATENT_DIM, EPOCHS, BATCH_SIZE)
print("Treinando AE para Markov...")
model_markov = train_ae(X_markov, INPUT_DIM, LATENT_DIM, EPOCHS, BATCH_SIZE)

def get_embeddings(model, X):
    model.eval()
    with torch.no_grad():
        _, z = model(torch.tensor(X))
    return z.numpy()

Z_iid = get_embeddings(model_iid, X_iid)
Z_markov = get_embeddings(model_markov, X_markov)

# 3. Métricas SCD
def runs_ratio(labels):
    n = len(labels)
    runs = 1
    for i in range(1, n):
        if labels[i] != labels[i-1]:
            runs += 1
    _, counts = np.unique(labels, return_counts=True)
    R_null = 1 + (n*n - np.sum(counts*counts)) / n
    return runs / R_null

def effrank(Z, eps=0.01):
    _, S, _ = np.linalg.svd(Z, full_matrices=False)
    return np.sum(S > eps * S[0])

def pc1_var(Z):
    return PCA(n_components=1).fit(Z).explained_variance_ratio_[0] * 100

def scd_metrics(Z, K, d_model=16):
    labels = KMeans(n_clusters=K, random_state=RANDOM_SEED, n_init=10).fit_predict(Z)
    Rr = runs_ratio(labels)
    er = effrank(Z)
    pc1 = pc1_var(Z)
    ndc6 = Rr < 0.5
    d_max = d_model // 3
    ndc6_strong = ndc6 or (er <= d_max and pc1 > 50.0)
    return {'K': K, 'R_ratio': Rr, 'effrank': er, 'PC1_var (%)': pc1, 'NDC-6 violated': ndc6, 'NDC-6-Strong': ndc6_strong}

print("\nCalculando para K = 2,4,6,8...")
res_iid = [scd_metrics(Z_iid, k, LATENT_DIM) for k in K_VALUES]
res_markov = [scd_metrics(Z_markov, k, LATENT_DIM) for k in K_VALUES]

df_iid = pd.DataFrame(res_iid)
df_markov = pd.DataFrame(res_markov)
df_iid['Dataset'] = 'i.i.d.'
df_markov['Dataset'] = 'Markov (persistence)'
df_comb = pd.concat([df_iid, df_markov], ignore_index=True)
df_comb = df_comb[['Dataset', 'K', 'R_ratio', 'effrank', 'PC1_var (%)', 'NDC-6 violated', 'NDC-6-Strong']]
df_comb.to_csv('scd_sensitivity_table.csv', index=False)
print("\nTabela salva em scd_sensitivity_table.csv")
print(df_comb.to_string(index=False))

# 4. Gerar figura TIFF com 1200 dpi
sns.set_style("whitegrid")
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(df_iid['K'], df_iid['R_ratio'], 'o-', label='i.i.d. (high entropy)', linewidth=2, markersize=8)
ax.plot(df_markov['K'], df_markov['R_ratio'], 's-', label='Markov (persistence)', linewidth=2, markersize=8)
ax.axhline(0.5, color='red', linestyle='--', label='NDC-6 threshold')
ax.set_xlabel('Number of clusters K')
ax.set_ylabel('R_ratio')
ax.set_title('Sensitivity of R_ratio to the number of clusters')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig('scd_sensitivity_Rratio.tiff', dpi=1200, format='tiff', pil_kwargs={'compression': 'tiff_lzw'})
plt.savefig('scd_sensitivity_Rratio.png', dpi=150)  # preview
plt.close()
print("Figura TIFF (1200 dpi) salva como scd_sensitivity_Rratio.tiff")
print("Pronto! Faça o download dos arquivos.")

Gerando dados...
Treinando AE para i.i.d...
Epoch 0, loss: 0.0828
Epoch 5, loss: 0.0617
Epoch 10, loss: 0.0573
Epoch 15, loss: 0.0565
Treinando AE para Markov...
Epoch 0, loss: 2.2941
Epoch 5, loss: 1.6419
Epoch 10, loss: 1.6379
Epoch 15, loss: 1.6627

Calculando para K = 2,4,6,8...

Tabela salva em scd_sensitivity_table.csv
             Dataset  K  R_ratio  effrank  PC1_var (%)  NDC-6 violated  NDC-6-Strong
              i.i.d.  2 0.998401       16    16.056152           False         False
              i.i.d.  4 0.995126       16    16.056152           False         False
              i.i.d.  6 0.995135       16    16.056152           False         False
              i.i.d.  8 0.994312       16    16.056152           False         False
Markov (persistence)  2 0.072650        3    97.376747            True          True
Markov (persistence)  4 0.071436        3    97.376747            True          True
Markov (persistence)  6 0.360982        3    97.376747            True        